# 05 — Policy Gradients & PPO

Policy gradient methods directly optimise the policy π(a|s;θ) instead of learning a value function first. This makes them naturally applicable to continuous action spaces and stochastic policies — and PPO (Proximal Policy Optimisation) is currently the most widely deployed deep RL algorithm.

## REINFORCE — the foundational idea (narrative)

REINFORCE (Williams, 1992) is the simplest policy gradient algorithm. The policy gradient theorem gives:
$$\nabla_\theta J(\theta) = \mathbb{E}_\pi\left[G_t \nabla_\theta \log \pi(A_t|S_t;\theta)\right]$$

Intuition: increase the log probability of actions that led to high returns, decrease it for actions that led to low returns.

**Why REINFORCE is not used in practice**:
- Requires complete episodes before updating
- Extremely high variance: G_t sums all future rewards, introducing enormous noise
- No baseline by default (the baseline-subtracted version — REINFORCE with baseline — is better but still slow)

REINFORCE is the conceptual ancestor of all policy gradient methods. The variance problem motivates actor-critic algorithms, and the instability of large updates motivates PPO.

## A2C and A3C (narrative)

**A2C** (Advantage Actor-Critic) addresses REINFORCE's variance by replacing G_t with the **advantage** A(s,a) = Q(s,a) - V(s) — how much better action a is than average in state s. A critic estimates V(s), the actor learns π(a|s).

**A3C** (Asynchronous Advantage Actor-Critic, Mnih et al. 2016) runs multiple agents in parallel environments to generate decorrelated experience — the idea behind parallel environment training that is standard today.

Both are superseded by PPO for most tasks. A2C/A3C are still worth knowing because:
- The actor-critic architecture is universal (PPO, SAC, and RLHF all use it)
- The advantage function is the core learning signal in PPO
- A3C's parallel environment idea is standard in modern training infrastructure

## PPO — why it dominates

PPO (Schulman et al., OpenAI 2017) solves the core instability problem of policy gradient methods: large policy updates can catastrophically degrade performance, and there's no way to know in advance how large is too large.

PPO uses a **clipped surrogate objective** that implicitly constrains how much the policy can change per update:
$$L^{CLIP}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)\hat{A}_t,\ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\right)\right]$$

Where `r_t(θ) = π(a_t|s_t;θ) / π(a_t|s_t;θ_old)` is the probability ratio. The clip at [1-ε, 1+ε] (typically ε=0.2) prevents the update from moving too far from the old policy.

This is simpler and more stable than the earlier TRPO (which used a hard KL constraint requiring second-order optimisation). PPO's simplicity is a feature: it scales to large networks, works across discrete and continuous action spaces, and is the backbone of RLHF fine-tuning.

In [ ]:
# PPO from scratch on CartPole (discrete) then LunarLander (continuous-ish)
# Full implementation: actor-critic network, advantage estimation (GAE), clipped objective

import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.actor = nn.Linear(hidden, act_dim)   # logits for discrete actions
        self.critic = nn.Linear(hidden, 1)         # state value estimate

    def forward(self, x):
        feat = self.shared(x)
        return self.actor(feat), self.critic(feat).squeeze(-1)

    def get_action(self, obs):
        logits, value = self(obs)
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), value

    def evaluate(self, obs, actions):
        logits, values = self(obs)
        dist = torch.distributions.Categorical(logits=logits)
        return dist.log_prob(actions), dist.entropy(), values

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """Generalised Advantage Estimation (Schulman et al., 2015).""""
    advantages = []
    gae = 0
    next_value = 0
    for r, v, d in zip(reversed(rewards), reversed(values), reversed(dones)):
        delta = r + gamma * next_value * (1 - d) - v
        gae = delta + gamma * lam * (1 - d) * gae
        advantages.insert(0, gae)
        next_value = v
    advantages = torch.tensor(advantages, dtype=torch.float32)
    returns = advantages + torch.tensor(values, dtype=torch.float32)
    return advantages, returns

class PPOAgent:
    def __init__(self, obs_dim, act_dim, lr=3e-4, clip_eps=0.2,
                 n_epochs=10, batch_size=64, entropy_coef=0.01):
        self.ac = ActorCritic(obs_dim, act_dim).to(device)
        self.optimizer = optim.Adam(self.ac.parameters(), lr=lr)
        self.clip_eps = clip_eps
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.entropy_coef = entropy_coef

    def update(self, obs, actions, old_log_probs, advantages, returns):
        obs = torch.FloatTensor(np.array(obs)).to(device)
        actions = torch.LongTensor(actions).to(device)
        old_log_probs = torch.FloatTensor(old_log_probs).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        returns = returns.to(device)

        total_loss = 0
        for _ in range(self.n_epochs):
            # Mini-batch updates
            idx = torch.randperm(len(obs))
            for start in range(0, len(obs), self.batch_size):
                mb_idx = idx[start:start+self.batch_size]
                mb_obs = obs[mb_idx]; mb_act = actions[mb_idx]
                mb_old_lp = old_log_probs[mb_idx]
                mb_adv = advantages[mb_idx]; mb_ret = returns[mb_idx]

                log_probs, entropy, values = self.ac.evaluate(mb_obs, mb_act)
                ratio = (log_probs - mb_old_lp).exp()

                # Clipped surrogate objective
                surr1 = ratio * mb_adv
                surr2 = ratio.clamp(1-self.clip_eps, 1+self.clip_eps) * mb_adv
                actor_loss = -torch.min(surr1, surr2).mean()

                # Value loss
                critic_loss = F.mse_loss(values, mb_ret)

                loss = actor_loss + 0.5 * critic_loss - self.entropy_coef * entropy.mean()
                self.optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.ac.parameters(), 0.5)
                self.optimizer.step()
                total_loss += loss.item()

        return total_loss

# ── Training loop ─────────────────────────────────────────────────────────
def train_ppo(env_id="CartPole-v1", n_steps_total=200_000, rollout_len=2048):
    env = gym.make(env_id)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.n
    agent = PPOAgent(obs_dim, act_dim)

    obs, _ = env.reset(seed=42)
    episode_rewards = []
    ep_reward = 0
    all_returns = []

    for step in range(0, n_steps_total, rollout_len):
        # Collect rollout
        obs_buf, act_buf, lp_buf, rew_buf, val_buf, done_buf = [], [], [], [], [], []

        for _ in range(rollout_len):
            with torch.no_grad():
                obs_t = torch.FloatTensor(obs).to(device)
                action, log_prob, value = agent.ac.get_action(obs_t.unsqueeze(0))
                action = action.item(); log_prob = log_prob.item(); value = value.item()

            next_obs, reward, term, trunc, _ = env.step(action)
            done = term or trunc

            obs_buf.append(obs); act_buf.append(action); lp_buf.append(log_prob)
            rew_buf.append(reward); val_buf.append(value); done_buf.append(float(done))

            obs = next_obs; ep_reward += reward
            if done:
                episode_rewards.append(ep_reward)
                ep_reward = 0
                obs, _ = env.reset()

        advantages, returns = compute_gae(rew_buf, val_buf, done_buf)
        agent.update(obs_buf, act_buf, lp_buf, advantages, returns)
        all_returns.extend(rew_buf)

        if len(episode_rewards) >= 10:
            avg = np.mean(episode_rewards[-20:])
            print(f"  Step {step+rollout_len:7d} | Episodes: {len(episode_rewards):4d} | Avg return: {avg:.1f}")

    env.close()
    return episode_rewards

print("Training PPO on CartPole-v1...")
returns = train_ppo("CartPole-v1", n_steps_total=100_000, rollout_len=1024)

w = 20
plt.figure(figsize=(9, 4))
plt.plot(returns, alpha=0.3, color='steelblue')
if len(returns) >= w:
    plt.plot(np.convolve(returns, np.ones(w)/w, mode='valid'), color='steelblue', label=f'{w}-ep avg')
plt.axhline(195, color='green', linestyle='--', alpha=0.7, label='Solved (195)')
plt.xlabel('Episode'); plt.ylabel('Return'); plt.title('PPO on CartPole-v1', fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('/tmp/ppo_cartpole.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"\nFinal 20-episode average: {np.mean(returns[-20:]):.1f}")


## PPO in production

PPO is the algorithm behind:
- **OpenAI Five** (Dota 2) and **AlphaStar** (StarCraft II) — both used PPO-based training at scale
- **InstructGPT and ChatGPT** RLHF fine-tuning (OpenAI, 2022)
- **Anthropic's RLHF pipeline** for Claude
- Most robotics locomotion baselines (though SAC is often preferred for sample efficiency)

The RLHF use is covered in depth in NB 12–13. The key architectural difference in RLHF: the policy being trained is an LLM, the action space is the vocabulary, and the reward signal comes from a trained reward model rather than an environment.